In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Session 0: Load and Analyze CSV File Structures

In [3]:
from pathlib import Path

# Get all CSV files from data_org (exclude metadata files)
data_org_path = Path('..') / 'data_org'
csv_files = sorted([f for f in data_org_path.glob('*.csv') if f.name != 'split_assignment.csv'])

# Load all files and store metadata
file_structure = {}
file_errors = []

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        file_structure[csv_file.name] = {
            'columns': list(df.columns),
            'num_columns': len(df.columns),
            'num_rows': len(df),
        }
    except Exception as e:
        file_errors.append({'Filename': csv_file.name, 'Error': str(e)})

# Create summary dataframe
file_summary_df = pd.DataFrame([
    {
        'Filename': filename,
        'Num_Rows': info['num_rows'],
        'Num_Columns': info['num_columns'],
        'Sample_Columns': ', '.join(info['columns'][:3]) + '...'
    }
    for filename, info in sorted(file_structure.items())
]).reset_index(drop=True)

print(f"✓ Successfully loaded: {len(file_structure)}/{len(csv_files)} files\n")
if file_errors:
    print("Errors encountered:")
    display(pd.DataFrame(file_errors))
else:
    print("✓ No errors\n")

print(f"Overall Statistics:")
print(f"  Total Files: {len(file_structure)}")
print(f"  Total Rows: {file_summary_df['Num_Rows'].sum()}")
print(f"  Avg Rows per File: {file_summary_df['Num_Rows'].mean():,.0f}\n")

display(file_summary_df)

✓ Successfully loaded: 0/0 files

✓ No errors

Overall Statistics:
  Total Files: 0


KeyError: 'Num_Rows'

In [ ]:
# Analyze column consistency across files
column_structure = []

for filename, info in sorted(file_structure.items()):
    columns = info['columns']
    column_structure.append({
        'Filename': filename,
        'Num_Columns': len(columns),
        'All_Columns': ' | '.join(columns)
    })

column_structure_df = pd.DataFrame(column_structure)

# Identify unique column structures
unique_structures = column_structure_df['All_Columns'].nunique()
print(f"Column Structure Analysis:")
print(f"  Unique structures found: {unique_structures}\n")

display(column_structure_df)

Column Structure Analysis:
  Unique structures found: 4



,Filename,Num_Columns,All_Columns
0,102_A2_y.csv,4,發生時間 | 發生地點 | 死亡受傷人數 | 車種
1,103_A2_y.csv,4,發生時間 | 發生地點 | 死亡受傷人數 | 車種
2,104_A2_y.csv,4,發生時間 | 發生地點 | 死亡受傷人數 | 車種
3,105_A2_y.csv,4,發生時間 | 發生地點 | 死亡受傷人數 | 車種
4,106_A2_y.csv,4,發生時間 | 發生地點 | 死亡受傷人數 | 車種
...,...,...,...
60,115_A2_m01.csv,51,發生年度 | 發生月份 | 發生日期 | 發生時間 | 事故類別名稱 | 處理單位名稱警局層...
61,115_A2_m02.csv,51,發生年度 | 發生月份 | 發生日期 | 發生時間 | 事故類別名稱 | 處理單位名稱警局層...
62,115_A2_m03.csv,51,發生年度 | 發生月份 | 發生日期 | 發生時間 | 事故類別名稱 | 處理單位名稱警局層...
63,115_A2_m04.csv,51,發生年度 | 發生月份 | 發生日期 | 發生時間 | 事故類別名稱 | 處理單位名稱警局層...


In [ ]:
# Group files by their column structure
structure_groups = column_structure_df.groupby('All_Columns')['Filename'].apply(list).reset_index()
structure_groups.columns = ['Column_Structure', 'Files']
structure_groups['Num_Files'] = structure_groups['Files'].apply(len)
structure_groups = structure_groups[['Num_Files', 'Column_Structure', 'Files']].sort_values('Num_Files', ascending=False)

print(f"Files Grouped by Column Structure ({len(structure_groups)} unique structures):\n")
display(structure_groups[['Num_Files', 'Column_Structure']])

Files Grouped by Column Structure (4 unique structures):



,Num_Files,Column_Structure
0,41,發生年度 | 發生月份 | 發生日期 | 發生時間 | 事故類別名稱 | 處理單位名稱警局層...
1,12,發生年度 | 發生月份 | 發生日期 | 發生時間 | 事故類別名稱 | 處理單位名稱警局層...
3,7,發生時間 | 發生地點 | 死亡受傷人數 | 車種 | 經度 | 緯度
2,5,發生時間 | 發生地點 | 死亡受傷人數 | 車種


## Session 1: Train/Test/Validation Split Strategy

In [ ]:
# Extract year from filename and assign to split
split_data = []

for filename in file_structure.keys():
    year = int(filename.split('_')[0])
    
    if year <= 109:
        split = 'Training'
        period = '2013-2020 (ROC yr 102-109)'
    elif year <= 112:
        split = 'Validation'
        period = '2021-2023 (ROC yr 110-112)'
    else:
        split = 'Testing'
        period = '2024-2026 (ROC yr 113-115)'
    
    split_data.append({
        'Filename': filename,
        'Year': year,
        'Split': split,
        'Period': period,
        'Num_Rows': file_structure[filename]['num_rows'],
        'Num_Columns': file_structure[filename]['num_columns'],
    })

# Create and sort by year
split_assignment_df = pd.DataFrame(split_data).sort_values('Year').reset_index(drop=True)

print("Dataset Split Assignment (Temporal Strategy):\n")
print("Rationale:")
print("  - Training (Yrs 102-109): Historical data to learn patterns")
print("  - Validation (Yrs 110-112): Intermediate data for model tuning")
print("  - Testing (Yrs 113-115): Recent data for unbiased evaluation")
print("  - Prevents data leakage: Test data chronologically after training/validation\n")

display(split_assignment_df)

Dataset Split Assignment (Temporal Strategy):

Rationale:
  - Training (Yrs 102-109): Historical data to learn patterns
  - Validation (Yrs 110-112): Intermediate data for model tuning
  - Testing (Yrs 113-115): Recent data for unbiased evaluation
  - Prevents data leakage: Test data chronologically after training/validation



,Filename,Year,Split,Period,Num_Rows,Num_Columns
0,102_A2_y.csv,102,Training,2013-2020 (ROC yr 102-109),276523,4
1,103_A2_y.csv,103,Training,2013-2020 (ROC yr 102-109),306076,4
2,104_A2_y.csv,104,Training,2013-2020 (ROC yr 102-109),303779,4
3,105_A2_y.csv,105,Training,2013-2020 (ROC yr 102-109),304003,4
4,106_A2_y.csv,106,Training,2013-2020 (ROC yr 102-109),295394,4
...,...,...,...,...,...,...
60,115_A2_m01.csv,115,Testing,2024-2026 (ROC yr 113-115),80598,51
61,115_A2_m02.csv,115,Testing,2024-2026 (ROC yr 113-115),67408,51
62,115_A2_m03.csv,115,Testing,2024-2026 (ROC yr 113-115),75490,51
63,115_A2_m04.csv,115,Testing,2024-2026 (ROC yr 113-115),50688,51


In [ ]:
# Comprehensive split summary
split_summary = split_assignment_df.groupby('Split').agg({
    'Filename': 'count',
    'Num_Rows': ['sum', 'mean', 'min', 'max'],
    'Year': ['min', 'max'],
    'Num_Columns': 'first'
}).round(0).astype(int)

split_summary.columns = ['Num_Files', 'Total_Rows', 'Avg_Rows', 'Min_Rows', 'Max_Rows', 'Year_Min', 'Year_Max', 'Num_Columns']

# Calculate percentages
total_rows = split_assignment_df['Num_Rows'].sum()
split_summary['Pct_of_Total'] = (
    split_assignment_df.groupby('Split')['Num_Rows'].sum() / total_rows * 100
).round(1)

print("Split Summary Statistics:\n")
display(split_summary)

# Create distribution dataframe
distribution_df = pd.DataFrame([
    {
        'Split': split_type,
        'Num_Files': len(split_assignment_df[split_assignment_df['Split'] == split_type]),
        'Total_Rows': split_assignment_df[split_assignment_df['Split'] == split_type]['Num_Rows'].sum(),
        'Percentage': f"{split_assignment_df[split_assignment_df['Split'] == split_type]['Num_Rows'].sum() / total_rows * 100:.1f}%"
    }
    for split_type in ['Training', 'Validation', 'Testing']
])

print("\nData Distribution:\n")
display(distribution_df)

Split Summary Statistics:



,Num_Files,Total_Rows,Avg_Rows,Min_Rows,Max_Rows,Year_Min,Year_Max,Num_Columns,Pct_of_Total
Split,,,,,,,,,
Testing,29,2066262,71250,1845,85501,113,115,51,31.0
Training,10,2505388,250539,166487,318860,102,109,4,37.5
Validation,26,2103476,80903,59456,182338,110,112,6,31.5



Data Distribution:



,Split,Num_Files,Total_Rows,Percentage
0,Training,10,2505388,37.5%
1,Validation,26,2103476,31.5%
2,Testing,29,2066262,31.0%


In [ ]:
# Save for manual folder organization
output_path = Path('..') / 'data_org' / 'split_assignment.csv'
split_assignment_df.to_csv(output_path, index=False)

print(f"✓ Split assignment saved to: {output_path}")
print("\nNext steps - Organize files into folders:")
print("  mkdir data_org/Training")
print("  mkdir data_org/Validation")
print("  mkdir data_org/Testing")
print("\nThen move files according to split_assignment.csv")

✓ Split assignment saved to: ..\data_org\split_assignment.csv

Next steps - Organize files into folders:
  mkdir data_org/Training
  mkdir data_org/Validation
  mkdir data_org/Testing

Then move files according to split_assignment.csv


## Session 2: Export All Data Structures to CSV

In [ ]:
from pathlib import Path

# Create output directory for exports
output_dir = Path('..') / 'data_org' / 'analysis_export'
output_dir.mkdir(parents=True, exist_ok=True)

# Function to save dataframe with section title header
def save_df_with_header(df, filename, section_title):
    filepath = output_dir / filename
    
    # Write section title as header comment (UTF-8-sig includes BOM for proper encoding detection)
    with open(filepath, 'w', encoding='utf-8-sig') as f:
        f.write(f"# {section_title}\n")
        f.write(f"# Export Date: {pd.Timestamp.now()}\n")
        f.write("#\n")
    
    # Append dataframe with UTF-8 encoding (BOM not needed for append mode)
    df.to_csv(filepath, mode='a', index=False, encoding='utf-8')
    return filepath

# Save all dataframes with section labels
exports = [
    (file_summary_df, 'Session_0_File_Summary.csv', 'Session 0: File Summary - Overview of all CSV files'),
    (column_structure_df, 'Session_0_Column_Structure.csv', 'Session 0: Column Structure Analysis - Detailed columns for each file'),
    (structure_groups[['Num_Files', 'Column_Structure']], 'Session_0_Structure_Groups.csv', 'Session 0: Files Grouped by Column Structure'),
    (split_assignment_df, 'Session_1_Split_Assignment.csv', 'Session 1: Train/Test/Validation Split Assignment'),
    (split_summary, 'Session_1_Split_Summary.csv', 'Session 1: Split Summary Statistics'),
    (distribution_df, 'Session_1_Distribution.csv', 'Session 1: Data Distribution Breakdown'),
]

print("Exporting all dataframes with section headers:\n")
for df, filename, title in exports:
    filepath = save_df_with_header(df, filename, title)
    print(f"✓ Saved: {filename}")
    print(f"  Location: {filepath}\n")

print(f"All exports saved to: {output_dir}")

Exporting all dataframes with section headers:

✓ Saved: Session_0_File_Summary.csv
  Location: ..\data_org\analysis_export\Session_0_File_Summary.csv

✓ Saved: Session_0_Column_Structure.csv
  Location: ..\data_org\analysis_export\Session_0_Column_Structure.csv

✓ Saved: Session_0_Structure_Groups.csv
  Location: ..\data_org\analysis_export\Session_0_Structure_Groups.csv

✓ Saved: Session_1_Split_Assignment.csv
  Location: ..\data_org\analysis_export\Session_1_Split_Assignment.csv

✓ Saved: Session_1_Split_Summary.csv
  Location: ..\data_org\analysis_export\Session_1_Split_Summary.csv

✓ Saved: Session_1_Distribution.csv
  Location: ..\data_org\analysis_export\Session_1_Distribution.csv

All exports saved to: ..\data_org\analysis_export


In [ ]:
# Create a comprehensive master file with all sections
master_filepath = output_dir / 'MASTER_Analysis_Report.csv'

with open(master_filepath, 'w', encoding='utf-8-sig') as f:
    f.write("# MASTER FILE STRUCTURE ANALYSIS REPORT\n")
    f.write(f"# Generated: {pd.Timestamp.now()}\n")
    f.write("# This file contains all analysis sections with full column structures\n")
    f.write("#\n\n")

section_data = [
    ('SESSION 0: File Summary', file_summary_df),
    ('SESSION 0: Column Structure Details', column_structure_df),
    ('SESSION 0: Structure Groups (Files by Column Structure)', structure_groups[['Num_Files', 'Column_Structure']]),
    ('SESSION 1: Split Assignment', split_assignment_df),
    ('SESSION 1: Split Summary Statistics', split_summary.reset_index()),
    ('SESSION 1: Data Distribution', distribution_df),
]

with open(master_filepath, 'a', encoding='utf-8') as f:
    for section_title, df in section_data:
        f.write(f"\n## {section_title}\n")
        f.write("#\n")
        df.to_csv(f, index=False, encoding='utf-8')
        f.write("\n")

print(f"\n✓ Master report created: MASTER_Analysis_Report.csv")
print(f"\nThis file contains:")
print(f"  • All 6 dataframes with section headers")
print(f"  • Complete column structures (no truncation)")
print(f"  • Timestamp of export")
print(f"\nLocation: {master_filepath}")


✓ Master report created: MASTER_Analysis_Report.csv

This file contains:
  • All 6 dataframes with section headers
  • Complete column structures (no truncation)
  • Timestamp of export

Location: ..\data_org\analysis_export\MASTER_Analysis_Report.csv
